# Text Classification from Scratch

Logistic regression and three Naive Bayes variants implemented with NumPy only —
no scikit-learn for the models or the metrics — then applied to spam email
detection and phishing URL detection.

Two datasets, deliberately different in shape:

- **Email spam**, 5,172 messages × 3,000 word-count features. High-dimensional
  and nearly linearly separable: **94.9% accuracy**.
- **Phishing URLs**, 11,430 URLs where the features have to be engineered from
  the URL string. Much harder: **72.9% accuracy**, and the feature-engineering
  story is the point.

**Headline result:** on the URL task, going from 3 hand-picked features to 9
lifts accuracy from 59.1% to 72.9%, and F1 from 0.35 to 0.71 — the model never
changed, only the features.

In [ ]:
from pathlib import Path

# Data is resolved relative to the repository root, so the notebook runs the
# same whether Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'

## 1. Bag of words

`CountVectorizer` limited to the 15 most frequent tokens, on a 10-message
sample. Small enough to print the whole document-term matrix and see what a
bag-of-words representation is before using one at scale.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import re


df = pd.read_csv(DATA / 'emails_1.csv')
print(df.head())
print("\n")

df['status'] = df['status'].map({'spam': 1, 'ham': 0})
print(df[['text', 'status']].head())
print("\n")

vectorizer = CountVectorizer(lowercase=True,token_pattern=r'[a-zA-Z]+',max_features=15)

X_bow = vectorizer.fit_transform(df['text'])

feature_names = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(X_bow.toarray(), columns=feature_names)

final_df = pd.concat([bow_df, df['status']], axis=1)

print(final_df)

## 2. The email dataset

5,172 emails, already vectorised into 3,000 word-count columns plus a
`Prediction` label. Class balance is 3,672 ham to 1,500 spam — roughly 71/29,
imbalanced enough that accuracy alone would be misleading, so precision and
recall are reported throughout.

In [ ]:
import pandas as pd

filename = DATA / 'emails_2.csv.gz'

  
df_emails_2 = pd.read_csv(filename, index_col=0)


print(f"FIRST 5 SAMPLE '{filename}' ---")
print(df_emails_2.head())

  


### A stratified split, written out

An 80/20 split implemented directly: shuffle each class separately, take the
first 80% of each. Stratifying matters at 29% positives — an unstratified split
can shift the spam proportion by several points between train and test and move
the metrics more than a model change would.

In [ ]:
import pandas as pd
import numpy as np

filename = DATA / 'emails_2.csv.gz'


df = pd.read_csv(filename, index_col=0)




y = df['status']
X = df.drop('status', axis=1)
    


test_size = 0.20
random_state = 42


train_indices = []
test_indices = []

rng = np.random.RandomState(random_state)


for cls in y.unique():

    class_indices = y[y == cls].index.to_numpy()
    

    rng.shuffle(class_indices)
    

    split_point = int(len(class_indices) * (1 - test_size))
    

    train_indices.extend(class_indices[:split_point])
    test_indices.extend(class_indices[split_point:])

rng.shuffle(train_indices)
rng.shuffle(test_indices)


X_train = X.loc[train_indices]
X_test = X.loc[test_indices]
y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

print("\n--- Data Split Results ---")
print("X_train, X_test, y_train, y_test variables created.")
print(f"Training samples (X_train): {X_train.shape[0]}")
print(f"Test samples (X_test): {X_test.shape[0]}")

## 3. Metrics from their definitions

Accuracy, precision, recall, and F1 from the confusion-matrix counts, checked
against a five-element example that can be verified by hand.

The zero guards are the substantive part: a model predicting no positives has
undefined precision, and returning 0.0 rather than crashing means a degenerate
model still produces a comparable number.

In [ ]:
import numpy as np

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    correct_predictions = np.sum(y_true == y_pred)
    
    return correct_predictions / len(y_true)

def precision(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    
    fp = np.sum((y_true == 0) & (y_pred == 1))
    
    predicted_positives = tp + fp
    
    if predicted_positives == 0:
        return 0.0
        
    return tp / predicted_positives

def recall(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    actual_positives = tp + fn
    
    if actual_positives == 0:
        return 0.0
        
    return tp / actual_positives

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    
    denominator = prec + rec
    
    if denominator == 0:
        return 0.0
        
    return 2 * (prec * rec) / denominator


y_true = [0, 1, 1, 0, 1]
y_pred = [0, 1, 0, 0, 1]

print(f"y_true = {y_true}")
print(f"y_pred = {y_pred}\n")

print(f"Accuracy:  {accuracy(y_true, y_pred):.2f}")
print(f"Precision: {precision(y_true, y_pred):.2f}")
print(f"Recall:    {recall(y_true, y_pred):.2f}")
print(f"F1-Score:  {f1_score(y_true, y_pred):.2f}")

## 4. Logistic regression by gradient descent

Batch gradient descent on the log-loss, 4,000 iterations at learning rate 0.5.

`np.clip(z, -250, 250)` before the sigmoid prevents `exp` overflow. With 3,000
features, early iterations produce large activations, and unclipped this warns
and returns `nan`, after which the weights never recover.

**Result: 94.88% accuracy, 0.9186 precision, 0.9033 recall, 0.9109 F1.**

In [ ]:
import numpy as np
import pandas as pd

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    correct_predictions = np.sum(y_true == y_pred)
    return correct_predictions / len(y_true)

def precision(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    predicted_positives = tp + fp
    if predicted_positives == 0:
        return 0.0
    return tp / predicted_positives

def recall(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    actual_positives = tp + fn
    if actual_positives == 0:
        return 0.0
    return tp / actual_positives

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    denominator = prec + rec
    if denominator == 0:
        return 0.0
    return 2 * (prec * rec) / denominator

class LogisticRegression:
    
    def __init__(self, learning_rate=0.5, n_iters=4000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.n_iters):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)
            
            dw = (1 / m_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / m_samples) * np.sum(y_predicted - y)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted_probs = self._sigmoid(linear_model)
        y_predicted_labels = [1 if i > 0.5 else 0 for i in y_predicted_probs]
        return np.array(y_predicted_labels)

filename = DATA / 'emails_2.csv.gz'
df = pd.read_csv(filename, index_col=0)

y = df['Prediction']
X = df.drop('Prediction', axis=1)


test_size = 0.20
random_state = 42

train_indices = []
test_indices = []

rng = np.random.RandomState(random_state)

for cls in y.unique():
    class_indices = y[y == cls].index.to_numpy()
    
    rng.shuffle(class_indices)
    
    split_point = int(len(class_indices) * (1 - test_size))
    
    train_indices.extend(class_indices[:split_point])
    test_indices.extend(class_indices[split_point:])

rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train_df = X.loc[train_indices]
X_test_df = X.loc[test_indices]
y_train_s = y.loc[train_indices]
y_test_s = y.loc[test_indices]

X_train = X_train_df.values
X_test = X_test_df.values
y_train = y_train_s.values
y_test = y_test_s.values


model = LogisticRegression(learning_rate=0.5, n_iters=4000)

model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)

acc = accuracy(y_test, y_pred_test)
prec = precision(y_test, y_pred_test)
rec = recall(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print(f"ACCURACY: {acc:.4f}")
print(f"PRECISION: {prec:.4f}")
print(f"RECALL: {rec:.4f}")
print(f"F1: {f1:.4f}")

## 5. Multinomial Naive Bayes

The generative counterpart, fitted by counting instead of by descent.
Probabilities are summed in log space — 3,000 factors below 1 would underflow to
zero in linear space — and add-one smoothing keeps a word absent from a class
from zeroing the entire product.

**Result: 94.20% accuracy, 0.8681 precision, 0.9433 recall, 0.9042 F1.**

Nearly identical accuracy to logistic regression but a different error profile:
Naive Bayes catches more spam (0.943 vs 0.903 recall) at the cost of more false
positives (0.868 vs 0.919 precision). For spam filtering, precision is usually
worth more — a legitimate email in the spam folder costs more than a spam email
in the inbox — which favours logistic regression here despite the lower recall.

In [ ]:
import numpy as np
import pandas as pd

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    correct_predictions = np.sum(y_true == y_pred)
    return correct_predictions / len(y_true)

def precision(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    predicted_positives = tp + fp
    if predicted_positives == 0:
        return 0.0
    return tp / predicted_positives

def recall(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    actual_positives = tp + fn
    if actual_positives == 0:
        return 0.0
    return tp / actual_positives

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    denominator = prec + rec
    if denominator == 0:
        return 0.0
    return 2 * (prec * rec) / denominator

class MultinomialNB:
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_priors_ = None
        self.feature_log_probs_ = None
        self.classes_ = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        self.class_priors_ = np.zeros(n_classes)
        self.feature_log_probs_ = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            
            self.class_priors_[idx] = np.log(X_c.shape[0] / n_samples)
            
            word_counts_in_class = X_c.sum(axis=0)
            
            total_words_in_class = word_counts_in_class.sum()
            
            numerator = word_counts_in_class + self.alpha
            denominator = total_words_in_class + (self.alpha * n_features)
            
            self.feature_log_probs_[idx] = np.log(numerator / denominator)

    def predict(self, X):
        y_pred = []
        for x in X:
            scores = self.class_priors_ + np.dot(x, self.feature_log_probs_.T)
            
            predicted_class = self.classes_[np.argmax(scores)]
            y_pred.append(predicted_class)
            
        return np.array(y_pred)

filename = DATA / 'emails_2.csv.gz'

df = pd.read_csv(filename, index_col=0)


y = df['Prediction']
X = df.drop('Prediction', axis=1)


test_size = 0.20
random_state = 42

train_indices = []
test_indices = []

rng = np.random.RandomState(random_state)

for cls in y.unique():
    class_indices = y[y == cls].index.to_numpy()
    
    rng.shuffle(class_indices)
    
    split_point = int(len(class_indices) * (1 - test_size))
    
    train_indices.extend(class_indices[:split_point])
    test_indices.extend(class_indices[split_point:])

rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train_df = X.loc[train_indices]
X_test_df = X.loc[test_indices]
y_train_s = y.loc[train_indices]
y_test_s = y.loc[test_indices]

X_train = X_train_df.values
y_train = y_train_s.values
X_test = X_test_df.values
y_test = y_test_s.values

model_mnb = MultinomialNB(alpha=1.0)

model_mnb.fit(X_train, y_train)

y_pred_test = model_mnb.predict(X_test)

print(f"Accuracy:  {accuracy(y_test, y_pred_test):.4f}")
print(f"Precision: {precision(y_test, y_pred_test):.4f}")
print(f"Recall:    {recall(y_test, y_pred_test):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_test):.4f}")

## 6. Phishing URLs: three structural features

A different problem. The dataset is 11,430 raw URL strings, balanced 50/50, so
every feature has to be engineered from the string itself.

Starting with three counts: dots, slashes, and hyphens.

Accuracy lands at 54–68% — barely above the 50% base rate. Punctuation counts
alone do not separate phishing from legitimate URLs.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

filename = DATA / 'urls.csv'
df = pd.read_csv(filename)

df['nb_dots'] = df['url'].str.count(r'\.')
df['nb_slashes'] = df['url'].str.count(r'/')
df['nb_hyphens'] = df['url'].str.count(r'-')

df['status'] = df['status'].map({'phishing': 1, 'legitimate': 0})

features_df = df[['nb_dots', 'nb_slashes', 'nb_hyphens', 'status']]
print("--- Features Head ---")
print(features_df.head())

X = features_df.drop('status', axis=1)
y = features_df['status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print(f"\n--- Data split: {len(X_train)} train, {len(X_test)} test ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression (Normalized)": {
        "model": LogisticRegression(random_state=42),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled
    },
    "Multinomial Naive Bayes (Raw)": {
        "model": MultinomialNB(),
        "X_train": X_train,
        "X_test": X_test
    },
    "Gaussian Naive Bayes (Normalized)": {
        "model": GaussianNB(),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled
    },
    "Bernoulli Naive Bayes (Raw)": {
        "model": BernoulliNB(),
        "X_train": X_train,
        "X_test": X_test
    }
}

results = []

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

for model_name, config in models.items():
    model = config["model"]
    model.fit(config["X_train"], y_train)
    
    y_pred = model.predict(config["X_test"])
    
    metrics = get_metrics(y_test, y_pred)
    metrics["Model"] = model_name
    results.append(metrics)

print("\n--- Final Model Comparison ---")
comparison_df = pd.DataFrame(results)[["Model", "Accuracy", "Precision", "Recall", "F1-Score"]]
print(comparison_df)

### Three length-based features

Swapping in URL length, digit ratio, and longest word length. No better:
65–67%. Both feature sets are individually weak.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

filename = DATA / 'urls.csv' 
df = pd.read_csv(filename)

df['length_url'] = df['url'].str.len()

df['n_digits'] = df['url'].str.count(r'[0-9]')
df['ratio_digits_url'] = np.where(df['length_url'] > 0, df['n_digits'] / df['length_url'], 0)

def get_longest_word_len(url):
    words = re.split(r'[^a-zA-Z]+', url)
    valid_words = [w for w in words if w]
    if not valid_words:
        return 0
    return max(len(w) for w in valid_words)

df['longest_words_raw'] = df['url'].apply(get_longest_word_len)

df['status'] = df['status'].map({'phishing': 1, 'legitimate': 0})

features_df = df[['length_url', 'ratio_digits_url', 'longest_words_raw', 'status']]

print("\n--- First Output: First 5 rows of features dataframe ---")
print(features_df.head())

X = features_df.drop('status', axis=1)
y = features_df['status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42,
    stratify=y
)
print(f"\n--- Data split: {len(X_train)} train, {len(X_test)} test ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("--- Data normalized for Logistic Regression and GaussianNB ---")

print("--- Training and evaluating models... ---")

models = {
    "Logistic Regression (Normalized)": {
        "model": LogisticRegression(random_state=42),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled
    },
    "Gaussian Naive Bayes (Normalized)": {
        "model": GaussianNB(),
        "X_train": X_train_scaled, 
        "X_test": X_test_scaled
    },
    "Multinomial Naive Bayes (Raw)": {
        "model": MultinomialNB(),
        "X_train": X_train,
        "X_test": X_test
    },
    "Bernoulli Naive Bayes (Raw)": {
        "model": BernoulliNB(),
        "X_train": X_train,
        "X_test": X_test
    }
}

results = []

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

for model_name, config in models.items():
    model = config["model"]
    model.fit(config["X_train"], y_train)
    
    y_pred = model.predict(config["X_test"])
    
    metrics = get_metrics(y_test, y_pred)
    metrics["Model"] = model_name
    results.append(metrics)

print("\n--- Second Output: Final Model Comparison Table ---")
comparison_df = pd.DataFrame(results)[["Model", "Accuracy", "Precision", "Recall", "F1-Score"]]
print(comparison_df)

### Where the signal actually is

Plotting each candidate feature against the label, before fitting anything else.

In [ ]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


def count_digits_in_hostname(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
            
        if hostname:
            digits = re.findall(r'[0-9]', hostname)
            return len(digits)
        else:
            return 0
    except Exception:
        return 0

def is_ip_address(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            return 0
        if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', hostname):
            return 1
    except Exception:
        return 0
    return 0

def is_com_tld(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
            
        if hostname:
            if hostname.endswith('.com'):
                return 1
            else:
                return 0
        else:
            return 0
    except Exception:
        return 0


filename = DATA / 'urls.csv'
df = pd.read_csv(filename)


df['digits_in_hostname'] = df['url'].apply(count_digits_in_hostname)
df['is_ip_address'] = df['url'].apply(is_ip_address)
df['is_com_tld'] = df['url'].apply(is_com_tld)


    
df['status_mapped'] = df['status'].map({'phishing': 1, 'legitimate': 0})
df['Status Label'] = df['status'].map({
    'phishing': 'Phishing (1)', 
    'legitimate': 'Legitimate (0)'
})

df.dropna(subset=['status_mapped', 'Status Label'], inplace=True)

print("Data processed. Starting to plot...")



sns.set_theme(style="whitegrid")


plt.figure(figsize=(8, 6))
sns.countplot(x='is_ip_address', hue='Status Label', data=df)
plt.title('Distribution of "is_ip_address" by Status', fontsize=16)
plt.xlabel('Is Hostname an IP Address?', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['No (0)', 'Yes (1)'], fontsize=10)
plt.legend(title='Status')
plt.tight_layout()



plt.figure(figsize=(8, 6))
sns.countplot(x='is_com_tld', hue='Status Label', data=df)
plt.title('Distribution of "is_com_tld" by Status', fontsize=16)
plt.xlabel('Does Hostname end in .com?', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['No (0)', 'Yes (1)'], fontsize=10)
plt.legend(title='Status')
plt.tight_layout()

plt.figure(figsize=(12, 7))


order = df['digits_in_hostname'].value_counts().index
if len(order) > 20:
    order = order[:20]

sns.countplot(x='digits_in_hostname', hue='Status Label', data=df, order=order)
plt.title('Distribution of "digits_in_hostname" by Status (Top 20)', fontsize=16)
plt.xlabel('Number of Digits in Hostname', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(fontsize=10)
plt.legend(title='Status')
plt.tight_layout()


plt.show()

### Three host-based features

Digits in the hostname, whether the host is a bare IP address, and whether the
TLD is `.com`.

The aggregate accuracy is the worst yet (59.1%), but the precision tells a
different story: **0.86 for logistic regression and 0.94 for Gaussian Naive
Bayes**, at recall of 0.22 and 0.14. These features are *highly* specific and
*rarely* present — a hostname containing digits is almost certainly phishing,
but most phishing hostnames contain no digits. Strong evidence when it fires,
silent otherwise.

In [ ]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

def count_digits_in_hostname(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
            
        if hostname:
            digits = re.findall(r'[0-9]', hostname)
            return len(digits)
        else:
            return 0
    except Exception:
        return 0

def is_ip_address(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            return 0
        if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', hostname):
            return 1
    except Exception:
        return 0
    return 0

def is_com_tld(url):
    
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
            
        if hostname:
         
            if hostname.endswith('.com'):
                return 1
            else:
                return 0
        else:
            return 0
    except Exception:
        return 0

filename = DATA / 'urls.csv'
df = pd.read_csv(filename)
    

df['digits_in_hostname'] = df['url'].apply(count_digits_in_hostname)
df['is_ip_address'] = df['url'].apply(is_ip_address)
df['is_com_tld'] = df['url'].apply(is_com_tld) 

df['status'] = df['status'].map({'phishing': 1, 'legitimate': 0})


features_df = df[['digits_in_hostname', 'is_ip_address', 'is_com_tld', 'status']]

print("\n--- First Output: First 5 rows of features dataframe ---")
print(features_df.head())

X = features_df.drop('status', axis=1)
y = features_df['status']

if y.isnull().any():
    print("Warning: Some 'status' values were unknown and have been ignored.")
    X = X[y.notnull()]
    y = y[y.notnull()]
    y = y.astype(int)

if X.empty or y.empty:
    print("Error: No valid data remaining for training after cleaning 'status'.")
    exit()
    
if len(np.unique(y)) < 2:
    print(f"Error: Only one class present in data. Cannot train or stratify split.")
    stratify_option = None
else:
    stratify_option = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=stratify_option
)
print(f"\n--- Data split: {len(X_train)} train, {len(X_test)} test ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("--- Data normalized ---")

print("--- Training and evaluating models... ---")

models = {
    "Logistic Regression (Normalized)": {
        "model": LogisticRegression(random_state=42),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled
    },
    "Gaussian Naive Bayes (Normalized)": {
        "model": GaussianNB(),
        "X_train": X_train_scaled, 
        "X_test": X_test_scaled
    },
    "Bernoulli Naive Bayes (Raw)": {
        "model": BernoulliNB(),
        "X_train": X_train,
        "X_test": X_test
    },
    "Multinomial Naive Bayes (Raw)": {
        "model": MultinomialNB(),
        "X_train": X_train,
        "X_test": X_test
    }
}

results = []

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

for model_name, config in models.items():
    model = config["model"]
    model.fit(config["X_train"], y_train)
    y_pred = model.predict(config["X_test"])
    
    metrics = get_metrics(y_test, y_pred)
    metrics["Model"] = model_name
    results.append(metrics)

print("\n--- Second Output: Final Model Comparison ---")
comparison_df = pd.DataFrame(results)[["Model", "Accuracy", "Precision", "Recall", "F1-Score"]]
print(comparison_df.to_string(index=False, float_format="%.4f"))

## 7. All nine features together

Combining the punctuation, length, and host-based groups.

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| **Logistic regression** | **0.7288** | 0.7571 | 0.6737 | **0.7130** |
| Multinomial NB | 0.6780 | 0.7887 | 0.4864 | 0.6017 |
| Bernoulli NB | 0.6654 | 0.6882 | 0.6045 | 0.6437 |
| Gaussian NB | 0.6238 | 0.9275 | 0.2686 | 0.4166 |

Accuracy rises from 59.1% to 72.9% and F1 from 0.35 to 0.71 — more than double.
The three feature groups are complementary: each is weak alone because each
covers a different subset of phishing URLs.

Gaussian Naive Bayes is the instructive failure. It has the best precision of
any model here (0.9275) and the worst F1 (0.4166), because its independence
assumption is badly violated — `length_url`, `longest_words_raw`, and
`ratio_digits_url` are all functions of the same string. It ends up confident
and silent, firing on 27% of positives.

In [ ]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')



def count_digits_in_hostname(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
        if hostname:
            digits = re.findall(r'[0-9]', hostname)
            return len(digits)
        else:
            return 0
    except Exception:
        return 0

def is_ip_address(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            return 0
        if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', hostname):
            return 1
    except Exception:
        return 0
    return 0

def is_com_tld(url):
    try:
        hostname = urlparse(str(url)).hostname
        if hostname is None:
            hostname = str(url).split('/')[0]
            hostname = re.sub(r':\d+', '', hostname)
        if hostname:
            return 1 if hostname.endswith('.com') else 0
        else:
            return 0
    except Exception:
        return 0

def get_longest_word_len(url):
    try:
        words = re.split(r'[^a-zA-Z]+', str(url))
        valid_words = [w for w in words if w]
        if not valid_words:
            return 0
        return max(len(w) for w in valid_words)
    except Exception:
        return 0



filename = DATA / 'urls.csv'
df = pd.read_csv(filename)
    

df['digits_in_hostname'] = df['url'].apply(count_digits_in_hostname)
df['is_ip_address'] = df['url'].apply(is_ip_address)
df['is_com_tld'] = df['url'].apply(is_com_tld)
df['length_url'] = df['url'].str.len()
df['longest_words_raw'] = df['url'].apply(get_longest_word_len)
df['nb_dots'] = df['url'].str.count(r'\.')
df['nb_slashes'] = df['url'].str.count(r'/')
df['nb_hyphens'] = df['url'].str.count(r'-')
# Create the intermediate column for calculation
df['n_digits_total'] = df['url'].str.count(r'[0-9]')
# Create the final ratio feature
df['ratio_digits_url'] = np.where(df['length_url'] > 0, df['n_digits_total'] / df['length_url'], 0)

df['status'] = df['status'].map({'phishing': 1, 'legitimate': 0})


feature_columns = [
    'digits_in_hostname',
    'is_ip_address',
    'is_com_tld',
    'ratio_digits_url',
    'longest_words_raw',
    'length_url',
    'nb_dots',
    'nb_slashes',
    'nb_hyphens',
    'status'
]


features_df = df[feature_columns].copy()


features_df.dropna(inplace=True)


print("\n--- First Output: First 5 rows of 9-feature dataframe ---")
print(features_df.head())


features_df['status'] = features_df['status'].astype(int)

X = features_df.drop('status', axis=1)
y = features_df['status']

if X.empty or y.empty:
    print("Error: No valid data remaining for training.")
    exit()
    
if len(np.unique(y)) < 2:
    print(f"Error: Only one class present in data. Cannot train or stratify split.")
    stratify_option = None
else:
    stratify_option = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=stratify_option
)
print(f"\n--- Data split: {len(X_train)} train, {len(X_test)} test ---")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("--- Data normalized ---")



print("--- Training and evaluating models... ---")

models = {
    "Logistic Regression (Normalized)": {
        "model": LogisticRegression(random_state=42, max_iter=1000), # Added max_iter
        "X_train": X_train_scaled,
        "X_test": X_test_scaled
    },
    "Gaussian Naive Bayes (Normalized)": {
        "model": GaussianNB(),
        "X_train": X_train_scaled, 
        "X_test": X_test_scaled
    },
    "Bernoulli Naive Bayes (Raw)": {
        "model": BernoulliNB(),
        "X_train": X_train.values, # Use .values for consistency
        "X_test": X_test.values
    },
    "Multinomial Naive Bayes (Raw)": {
        "model": MultinomialNB(),
        "X_train": X_train.values,
        "X_test": X_test.values
    }
}

results = []

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

for model_name, config in models.items():
    model = config["model"]
    model.fit(config["X_train"], y_train)
    y_pred = model.predict(config["X_test"])
    
    metrics = get_metrics(y_test, y_pred)
    metrics["Model"] = model_name
    results.append(metrics)


print("\n--- Second Output: Final Model Comparison ---")
comparison_df = pd.DataFrame(results)[["Model", "Accuracy", "Precision", "Recall", "F1-Score"]]
print(comparison_df.to_string(index=False, float_format="%.4f"))

### Reading the learned weights

| Feature | Weight |
|---|---|
| `digits_in_hostname` | **+2.0934** |
| `length_url` | +0.9754 |
| `nb_hyphens` | −0.8099 |
| `ratio_digits_url` | +0.5023 |
| `nb_slashes` | +0.4453 |
| `longest_words_raw` | +0.2537 |
| `is_com_tld` | −0.1812 |
| `is_ip_address` | +0.0939 |
| `nb_dots` | +0.0774 |

`digits_in_hostname` dominates at more than twice the next weight, matching the
distribution plot above. Two signs are worth reading:

- **`is_com_tld` is negative** — a `.com` domain is weak evidence of legitimacy.
- **`nb_hyphens` is negative**, which is counterintuitive, since hyphens are a
  classic typosquatting marker (`paypal-secure-login.com`). With correlated
  features a single coefficient is not the feature's total effect: hyphens
  co-occur with long URLs, `length_url` already carries a large positive weight,
  and the hyphen coefficient partly corrects for that overlap. This is the
  standard hazard of reading correlated linear models one weight at a time.

In [ ]:
logistic_model = models["Logistic Regression (Normalized)"]["model"]
feature_names = X.columns
weights = logistic_model.coef_[0]
bias = logistic_model.intercept_[0]

weights_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight (Coefficient)': weights
})

weights_df['Absolute_Weight'] = weights_df['Weight (Coefficient)'].abs()
weights_df = weights_df.sort_values(by='Absolute_Weight', ascending=False)

print("\n--- Logistic Regression Model Weights ---")
print(weights_df[['Feature', 'Weight (Coefficient)']].to_string(index=False, float_format="%.4f"))

print("\n" + "-"*30)
print(f"Bias (Intercept): {bias:.4f}")
print("-"*30)